# 06 – LSTM (Day 9)

Same data and same cleaning as the baseline. **Number to beat: val macro-F1 0.696 (B-none).**

In [1]:
import sys
sys.path.append("..")   # notebooks/ is one level down, so let Python see the project root

import pandas as pd
from src.text import clean_text   # ONE shared cleaning function: no training/serving skew

In [2]:
splits = {name: pd.read_parquet(f"../data/processed/{name}.parquet")
          for name in ["train", "val", "test", "test_dedup"]}

N = "Consumer complaint narrative"
for name, part in splits.items():
    part["text_clean"] = part[N].map(clean_text)
    print(name, part.shape)

train (367414, 6)
val (142407, 6)
test (169189, 6)
test_dedup (87091, 6)


In [3]:
import tensorflow as tf
import keras

# 25,000 ≈ the baseline's 24,739 words (fair comparison); 350 = the Day 3 max length
vectorizer = keras.layers.TextVectorization(max_tokens=25_000, output_sequence_length=350)

# adapt = Keras's "fit": learn the vocabulary from TRAIN only, in batches to keep memory low
ds = tf.data.Dataset.from_tensor_slices(splits["train"]["text_clean"].values).batch(1024)
vectorizer.adapt(ds)

In [4]:
vocab = vectorizer.get_vocabulary()
print("Vocabulary size:", len(vocab))
print(vocab[:20])

Vocabulary size: 25000
['', '[UNK]', np.str_('redactedinfo'), np.str_('the'), np.str_('to'), np.str_('i'), np.str_('and'), np.str_('of'), np.str_('a'), np.str_('my'), np.str_('that'), np.str_('credit'), np.str_('this'), np.str_('in'), np.str_('is'), np.str_('account'), np.str_('on'), np.str_('not'), np.str_('information'), np.str_('have')]


In [10]:
#Finding short tickets 
train = splits["train"]
word_count = train['text_clean'].str.split().str.len()

short_text = train.loc[word_count<50,'text_clean'].iloc[0]
long_text = train.loc[word_count>350,'text_clean'].iloc[0]
unseen_text = splits['val']['text_clean'].sample(1, random_state=42).iloc[0]

texts = [short_text, long_text, unseen_text]
print(len(long_text.split()),len(unseen_text.split()))
print(len(short_text.split()),'words:',short_text)

432 281
39 words: my credit reports are inaccurate. these inaccuracies are causing creditors to deny me credit. you have the duty to report accurate information about consumers. please investigate these accounts and inquires and update these accounts accordingly to avoid future litigation.


In [11]:
#Encoding part 
ids = vectorizer(tf.constant(texts)).numpy()
print(ids.shape)

(3, 350)


In [12]:
vocab = vectorizer.get_vocabulary()

In [13]:
for name, row in zip(['short','long','unseen'],ids):
    print(name,"| real:",(row!=0).sum(),
    "|padded:",(row==0).sum(),
    '| unknown:',(row==1).sum(),)
    print(' back:', [str(vocab[i]) for i in row [:10]])

short | real: 39 |padded: 311 | unknown: 0
 back: ['my', 'credit', 'reports', 'are', 'inaccurate', 'these', 'inaccuracies', 'are', 'causing', 'creditors']
long | real: 350 |padded: 0 | unknown: 0
 back: ['after', 'reviewing', 'my', 'consumer', 'report', 'i', 'noticed', 'that', 'there', 'was']
unseen | real: 268 |padded: 82 | unknown: 0
 back: ['i', 'recently', 'received', 'a', 'copy', 'of', 'my', 'credit', 'report', 'i']


In [14]:
print(1 - word_count.clip(upper=350).mean() / 350)


0.5033515637089807


In [18]:
import joblib


label_order = sorted(train["label"].unique())
print(len(label_order), label_order)

baseline = joblib.load("../models/logreg_b.joblib")
print("Same as baseline:", list(baseline.classes_) == label_order)


11 ['Checking or savings account', 'Credit card', 'Credit reporting or other personal consumer reports', 'Debt collection', 'Debt or credit management', 'Money transfer, virtual currency, or money service', 'Mortgage', 'Payday loan, title loan, personal loan, or advance loan', 'Prepaid card', 'Student loan', 'Vehicle loan or lease']
Same as baseline: True


In [19]:
import json

with open("../models/label_order.json", "w") as f:
    json.dump(label_order, f, indent=2)

# Proof: load it back and compare
with open("../models/label_order.json") as f:
    loaded = json.load(f)
print("Saved correctly:", loaded == label_order)


Saved correctly: True


In [20]:
#Names to numbers
label_to_id = {label: i for i, label in enumerate(label_order)}
print(label_to_id)


{'Checking or savings account': 0, 'Credit card': 1, 'Credit reporting or other personal consumer reports': 2, 'Debt collection': 3, 'Debt or credit management': 4, 'Money transfer, virtual currency, or money service': 5, 'Mortgage': 6, 'Payday loan, title loan, personal loan, or advance loan': 7, 'Prepaid card': 8, 'Student loan': 9, 'Vehicle loan or lease': 10}


In [21]:
y = {}
for name, part in splits.items():
    mapped = part["label"].map(label_to_id)
    print(name, "NaN:", mapped.isna().sum())
    y[name] = mapped.to_numpy()


train NaN: 0
val NaN: 0
test NaN: 0
test_dedup NaN: 0


In [23]:
import numpy as np

counts = np.bincount(y['train'])
print("Box 4 Counts:", counts[4])
print("Original:",train['label'].value_counts()['Debt or credit management'])

Box 4 Counts: 969
Original: 969


In [24]:
back = [label_order[i] for i in y["val"][:5]]
print(back)
print(list(splits["val"]["label"][:5]))
print("Round trip OK:", back == list(splits["val"]["label"][:5]))


['Credit reporting or other personal consumer reports', 'Credit reporting or other personal consumer reports', 'Credit reporting or other personal consumer reports', 'Credit reporting or other personal consumer reports', 'Money transfer, virtual currency, or money service']
['Credit reporting or other personal consumer reports', 'Credit reporting or other personal consumer reports', 'Credit reporting or other personal consumer reports', 'Credit reporting or other personal consumer reports', 'Money transfer, virtual currency, or money service']
Round trip OK: True


## Day 10, Subtask 1: Baseline on truncated text

**Prediction (written before running):** truncated-LR val macro-F1 ≈ 0.70
(full-text LR = 0.696). Reason: only about 10% of complaints are longer than 350 words, so 90% of rows are unchanged, and the long ones keep their first 350 words. The cut-off ends are often legal boilerplate and sign-offs, so removing them may cut a little noise. I expect almost no change, maybe a tiny gain.

**Rule:** if it's within 0.005 of 0.696, I'll say the routing signal is mostly early.
If it drops by more than that, the end of long complaints matters.

**Caveat:** I truncate on spaces, while Keras also strips punctuation-only tokens, so "350 words" ≈ "350 Keras tokens" but they're not identical.


In [28]:
def first_350(text:str)->str:
    """Keep only the first 350 words (space-split), like the LSTM's view."""
    words = text.split()
    return " ".join(words[:350])

for name in ['train','val']:
    splits[name]['text_350'] = splits[name]['text_clean'].map(first_350)


In [29]:
print(splits["train"]["text_350"].str.split().str.len().max())   # expect 350


350


In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [32]:
vec_350 = TfidfVectorizer(min_df=5 , ngram_range=(1,1), sublinear_tf=True)
X_train_350 = vec_350.fit_transform(splits['train']['text_350'])
X_val_350 = vec_350.transform(splits['val']['text_350'])

#Sample Model, no weights

lr_350 = LogisticRegression(max_iter=1000)

lr_350.fit(X_train_350,splits['train']['label'])

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [33]:
import joblib
from sklearn.metrics import f1_score, accuracy_score

val = splits["val"]
y_true = val["label"]

# 1. Predictions from BOTH models
vec_b = joblib.load("../models/tfidf_b.joblib")
lr_b  = joblib.load("../models/logreg_b.joblib")
pred_full = lr_b.predict(vec_b.transform(val["text_clean"]))      # full text
pred_350  = lr_350.predict(X_val_350)                        # cut text

# 2. Overall scores
for name, p in [("full", pred_full), ("350", pred_350)]:
    print(name, "| macro-F1:", round(f1_score(y_true, p, average="macro"), 4),
                "| acc:", round(accuracy_score(y_true, p), 4))

# 3. Long rows only
long = (val["text_clean"].str.split().str.len() > 350).to_numpy()
print("Long val rows:", long.sum(), f"({long.mean():.1%})")
for name, p in [("full", pred_full), ("350", pred_350)]:
    print(name, "long macro-F1:", round(f1_score(y_true[long], p[long], average="macro"), 4))


full | macro-F1: 0.6956 | acc: 0.9157
350 | macro-F1: 0.6943 | acc: 0.9148
Long val rows: 12361 (8.7%)
full long macro-F1: 0.6736
350 long macro-F1: 0.6576


## Day 10, Subtask 2: LSTM gates on paper

**Task A** (c_old = 2.0, g = 0.5)

| Scenario | f | i | o | c_new | h_new |
|---|---|---|---|---|---|
| 1. Keep the memory | 0.9 | 0.1 | 0.5 | 0.9×2.0 + 0.1×0.5 = **1.850** | 0.5 × tanh(1.85) = **0.476** |
| 2. Overwrite | 0.1 | 0.9 | 0.5 | 0.1×2.0 + 0.9×0.5 = **0.650** | 0.5 × tanh(0.65) = **0.286** |

**Task B**
1. The **forget gate f** must stay close to 1, so "mortgage" isn't erased during the next 300 words.
2. Gates answer **how much** (a 0–1 dial), while g answers **what** (new content that can push the memory up or down, so it needs negative values).
3. 0.5³⁰⁰ ≈ 5×10⁻⁹¹, so repeated multiplication makes memory vanish. The LSTM adds new information to the kept memory, and with f ≈ 1 the cell state passes through almost unchanged (and so does the gradient), so it can remember across 300 steps.

**Task C:** 4 × [64 × (64 + 64) + 64] = 4 × 8,256 = **33,024** parameters.

**Future improvement (from Subtask 1):** head + tail truncation (the first ~250 + the last ~100 tokens), since the tails of long complaints carry some signal.


In [34]:
train = splits["train"]
print("Sorted by date?", train["Date received"].is_monotonic_increasing)

Sorted by date? False


In [35]:
train = splits["train"].sample(frac=1.0, random_state=42)        # shuffle ALL rows once
y_train = train["label"].map(label_to_id).to_numpy()              # labels in the NEW order

print("First 20k Credit reporting share:", (train["label"].iloc[:20000] == label_order[2]).mean())


First 20k Credit reporting share: 0.69035


In [39]:
#Conveyor Belt
AUTOTUNE = tf.data.AUTOTUNE
# TRAIN: shuffle → batch → text-to-IDs → prefetch

train_ds =(
    tf.data.Dataset.from_tensor_slices((train['text_clean'].values, y_train))
    .shuffle(20_000, seed=42)      # shuffle 20k rows at a time
    .batch(256)                     # batch 256 rows at a time
    .map(lambda x, y: (vectorizer(x), y), num_parallel_calls=AUTOTUNE)  # text → IDs
    .prefetch(AUTOTUNE)
)


In [40]:
# VAL: NO shuffle, so prediction i lines up with y_val[i] for macro-F1
val_ds = (
    tf.data.Dataset.from_tensor_slices((splits["val"]["text_clean"].values, y["val"]))
    .batch(256)
    .map(lambda x, y: (vectorizer(x), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

In [41]:
import math

for name, ds in [("train", train_ds), ("val", val_ds)]:
    for x, y_batch in ds.take(1):
        print(name, "| x:", x.shape, x.dtype, "| y:", y_batch.shape)
        print("   y[:10]:", y_batch[:10].numpy())

print("Batches per epoch:", math.ceil(len(train) / 256))


train | x: (256, 350) <dtype: 'int64'> | y: (256,)
   y[:10]: [2 2 2 2 2 2 2 2 2 0]
val | x: (256, 350) <dtype: 'int64'> | y: (256,)
   y[:10]: [2 2 2 2 5 5 0 2 2 2]
Batches per epoch: 1436


In [42]:
model = keras.Sequential([
    keras.Input(shape=(350,),dtype ='int64'),
    keras.layers.Embedding(input_dim=len(vocab),output_dim=64,mask_zero=True),
    keras.layers.LSTM(64),
    keras.layers.Dense(11,activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 350, 64)        │     1,600,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 11)             │           715 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,633,739 (6.23 MB)

 Trainable params: 1,633,739 (6.23 MB)

 Non-trainable params: 0 (0.00 B)

In [43]:
out = model(x)
print("Output shape:", out.shape)
print("Output sum (should be 1.0):", out.numpy().sum(axis=1)[:5])

Output shape: (256, 11)
Output sum (should be 1.0): [1.        1.        1.0000001 1.        0.9999999]


In [46]:
model.compile(
    loss = 'sparse_categorical_crossentropy',
    optimizer = keras.optimizers.Adam(learning_rate = 0.001,clipnorm=1.0),
    metrics = ['accuracy']
)

loss,acc = model.evaluate(train_ds.take(1))

print("Starting Loss:",round(loss,3))

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.0430 - loss: 2.4057
Starting Loss: 2.406


In [47]:
import time
from sklearn.metrics import f1_score

class MacroF1(keras.callbacks.Callback):
    def __init__(self, val_ds, y_val):
        super().__init__(); self.val_ds = val_ds; self.y_val = y_val

    def on_epoch_end(self, epoch, logs=None):
        probs = self.model.predict(self.val_ds, verbose=0)    # 11 probabilities per val row
        preds = probs.argmax(axis=1)                             # biggest slice → class ID
        assert len(preds) == len(self.y_val), "Val rows dropped!"
        f1 = f1_score(self.y_val, preds, average="macro")
        logs["val_macro_f1"] = f1                                      # pin it on the notice board 📌
        print(f"Epoch {epoch + 1}: val macro-F1 = {f1:.4f}")


In [48]:
cb = MacroF1(val_ds, y["val"])
cb.set_model(model)
logs = {}

start = time.perf_counter()
cb.on_epoch_end(0, logs)
print("Time:", round(time.perf_counter() - start, 1), "s")
print(logs)


Epoch 1: val macro-F1 = 0.0252
Time: 126.6 s
{'val_macro_f1': 0.025185012431630696}


In [49]:
small_ds = train_ds.take(40)

times,losses = [], []
for e in range(2):
    start = time.perf_counter()
    h = model.fit(small_ds,epochs=2)
    times.append(time.perf_counter() - start)
    losses.append(h.history["loss"][0])

print("Losses:",[round(l,3) for l in losses])
print("Times (s):", [round(t, 1) for t in times])


Epoch 1/2


c:\Users\srisu\OneDrive\Desktop\MachineLearning\Production Grade Projects\Support Ticket Routing\nlp\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


40/40 ━━━━━━━━━━━━━━━━━━━━ 31s 661ms/step - accuracy: 0.6651 - loss: 1.5128
Epoch 2/2
40/40 ━━━━━━━━━━━━━━━━━━━━ 26s 646ms/step - accuracy: 0.6930 - loss: 1.0599
Epoch 1/2
40/40 ━━━━━━━━━━━━━━━━━━━━ 26s 645ms/step - accuracy: 0.7104 - loss: 0.9232
Epoch 2/2
40/40 ━━━━━━━━━━━━━━━━━━━━ 26s 645ms/step - accuracy: 0.7202 - loss: 0.8613
Losses: [1.513, 0.923]
Times (s): [57.2, 51.9]


In [51]:
#NOTE: smoke loop ran epochs=2 per iteration by mistake (4 epochs total), so these times are 2×.
# Corrected from Keras's 645 ms/step: ≈17.6 min/epoch incl. validation, ≈2.9 h for 10 epochs.
per_batch = times[1] / 40                  # use the 2nd run (no warm-up)
full_epoch = per_batch * 1436 + 127           # train batches + callback time
print(f"Per batch: {per_batch:.2f} s")
print(f"Full epoch (with val): {full_epoch / 60:.1f} min")
print(f"10 epochs: {full_epoch * 10 / 3600:.1f} h")


Per batch: 1.30 s
Full epoch (with val): 33.2 min
10 epochs: 5.5 h


In [56]:
def build_model():
    model = keras.Sequential([
        keras.Input(shape=(350,), dtype="int64"),
        keras.layers.Embedding(input_dim=len(vocab), output_dim=64, mask_zero=True),
        keras.layers.LSTM(64),
        keras.layers.Dense(11, activation="softmax")
    ])

    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0),
        metrics=["accuracy"]
    )
    return model



In [57]:
model = build_model()

In [58]:
with open("../models/lstm_vocab.json", "w") as f:
    json.dump(vectorizer.get_vocabulary(), f)

with open("../models/lstm_vocab.json") as f:
    loaded_vocab = json.load(f)
print("Words saved:", len(loaded_vocab))      # expect 25000


Words saved: 25000


In [60]:
vec_loaded = keras.layers.TextVectorization(
    max_tokens=25_000, output_sequence_length= 350, vocabulary=loaded_vocab)

ids_a = vectorizer(tf.constant(texts)).numpy()
ids_b = vec_loaded(tf.constant(texts)).numpy()
assert (ids_a == ids_b).all(), "IDs differ!"
print("Identical IDs ✅", ids_a.shape)


Identical IDs ✅ (3, 350)


In [61]:
keras.utils.set_random_seed(42)       # same starting weights every run 🌱
model = build_model()                 # fresh, untrained model

callbacks = [
    MacroF1(val_ds, y["val"]),                                   # 1. writes val_macro_f1 FIRST
    keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max",
                                  patience=2, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("../models/lstm_best.keras", monitor="val_macro_f1",
                                    mode="max", save_best_only=True),
    keras.callbacks.BackupAndRestore(backup_dir="../models/lstm_backup"),
    keras.callbacks.CSVLogger("../models/lstm_log.csv"),
]

start = time.perf_counter()
history = model.fit(train_ds, epochs=15, callbacks=callbacks)    # no validation_data: MacroF1 does it
print(f"Total time: {(time.perf_counter() - start) / 3600:.2f} h")


Epoch 1/15


c:\Users\srisu\OneDrive\Desktop\MachineLearning\Production Grade Projects\Support Ticket Routing\nlp\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


1436/1436 ━━━━━━━━━━━━━━━━━━━━ 0s 588ms/step - accuracy: 0.7844 - loss: 0.6912Epoch 1: val macro-F1 = 0.2702
1436/1436 ━━━━━━━━━━━━━━━━━━━━ 964s 669ms/step - accuracy: 0.7844 - loss: 0.6912 - val_macro_f1: 0.2702
Epoch 2/15
1436/1436 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8523 - loss: 0.4629Epoch 2: val macro-F1 = 0.5345
1436/1436 ━━━━━━━━━━━━━━━━━━━━ 1621s 1s/step - accuracy: 0.8523 - loss: 0.4629 - val_macro_f1: 0.5345
Epoch 3/15
1436/1436 ━━━━━━━━━━━━━━━━━━━━ 0s 616ms/step - accuracy: 0.8886 - loss: 0.3605Epoch 3: val macro-F1 = 0.6569
1436/1436 ━━━━━━━━━━━━━━━━━━━━ 993s 691ms/step - accuracy: 0.8886 - loss: 0.3605 - val_macro_f1: 0.6569
Epoch 4/15
1436/1436 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9024 - loss: 0.3137Epoch 4: val macro-F1 = 0.6735
1436/1436 ━━━━━━━━━━━━━━━━━━━━ 2703s 2s/step - accuracy: 0.9024 - loss: 0.3137 - val_macro_f1: 0.6735
Epoch 5/15
1436/1436 ━━━━━━━━━━━━━━━━━━━━ 0s 592ms/step - accuracy: 0.9114 - loss: 0.2832Epoch 5: val macro-F1 = 0.6603
1436/

## Subtask 9: training results

| Epoch | Train loss | Train acc | Val macro-F1 | Time (s) |
|---|---|---|---|---|
| 1 | 0.691 | 0.784 | 0.270 | 964 |
| 2 | 0.463 | 0.852 | 0.534 | 1621 |
| 3 | 0.361 | 0.889 | 0.657 | 993 |
| **4** | **0.314** | **0.902** | **0.674** ⭐ | 2703 |
| 5 | 0.283 | 0.911 | 0.660 | 957 |
| 6 | 0.256 | 0.919 | 0.664 | 1546 |

- Best: epoch 4, val macro-F1 **0.6735** (saved to `lstm_best.keras`, restored by EarlyStopping)
- EarlyStopping fired after epoch 6 (patience 2); total time **2.44 h**
- vs truncated LR **0.694** → LSTM is **0.021 lower** (> ±0.005 rule) → LR wins
- Train loss kept falling while val F1 fell → overfitting after epoch 4
- Prediction was: ____ → right / wrong because ____


In [62]:
import os

lstm = keras.models.load_model("../models/lstm_best.keras")             # hint: notebook is in notebooks/ → "../"

# rebuild val from the SAVED vocab (vec_loaded), not the in-memory one
val_ds_loaded = (
    tf.data.Dataset.from_tensor_slices((splits["val"]["text_clean"].values, y["val"]))
    .batch(256)
    .map(lambda x, y_: (vec_loaded(x), y_), num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)                                                   # no shuffle! order must match y["val"]

probs = lstm.predict(val_ds_loaded, verbose=0)
preds = probs.argmax(axis=1)
f1 = f1_score(y['val'], preds, average="macro")
print(f"Reloaded val macro-F1: {f1:.4f}")
assert round(f1, 4) == 0.6735, "Saved file doesn't match!"

size_mb = os.path.getsize("../models/lstm_best.keras") / 1e6
print(f"File size: {size_mb:.1f} MB")


Reloaded val macro-F1: 0.6735
File size: 19.6 MB


In [65]:
import joblib
from sklearn.metrics import classification_report

# Step 0: save the fair benchmark
joblib.dump(vec_350, "../models/tfidf_b350.joblib")
joblib.dump(lr_350,  "../models/logreg_b350.joblib")

# Step 1: LSTM numbers → names
lstm_names = [label_order[i] for i in preds]          # hint: the LSTM's predictions

y_true = splits["val"]["label"]
rep_lr   = pd.DataFrame(classification_report(y_true, pred_350, output_dict=True)).T
rep_lstm = pd.DataFrame(classification_report(y_true, lstm_names, output_dict=True)).T

table = pd.DataFrame({
    "support":  rep_lr["support"],
    "LR F1":    rep_lr["f1-score"],
    "LSTM F1":  rep_lstm["f1-score"],
})
table = table.loc[label_order]                        # keep only the 11 classes
table["diff"] = table["LSTM F1"] - table["LR F1"]         # LSTM − LR
table.sort_values("diff").round(3)


c:\Users\srisu\OneDrive\Desktop\MachineLearning\Production Grade Projects\Support Ticket Routing\nlp\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\srisu\OneDrive\Desktop\MachineLearning\Production Grade Projects\Support Ticket Routing\nlp\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\srisu\OneDrive\Desktop\MachineLearning\Production Grade Projects\Support Ticket Routing\nlp\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill

,support,LR F1,LSTM F1,diff
Debt or credit management,358.0,0.275,0.000,-0.275
"Payday loan, title loan, personal loan, or advance loan",593.0,0.499,0.445,-0.054
Vehicle loan or lease,1013.0,0.628,0.617,-0.011
Mortgage,1260.0,0.854,0.849,-0.005
Credit reporting or other personal consumer reports,91690.0,0.955,0.959,0.004
Checking or savings account,9148.0,0.858,0.863,0.005
"Money transfer, virtual currency, or money service",24817.0,0.951,0.958,0.007
Credit card,3801.0,0.663,0.672,0.009
Student loan,1201.0,0.782,0.808,0.026
Debt collection,8124.0,0.610,0.638,0.027


Wins vs losses: The LSTM wins on 7 classes and loses on 4. Mortgage (−0.005), Credit reporting (+0.004) and Checking (+0.005) are within the ±0.005 tie rule, so the clear result is 5 wins and 3 losses.

Small classes: Mostly yes: its 3 clear losses are small classes (358, 593 and 1,013 tickets). But it's not purely size, because Prepaid card (402) is its biggest win (+0.038).


Debt or credit management: The LSTM scores 0.000 because it never predicts this class once. That single class costs it about 0.025 of macro-F1, which is more than the whole 0.0205 gap. On the other 10 classes, the LSTM is slightly ahead (0.741 vs 0.736).

In [75]:
import numpy as np

lr_pred = np.array(pred_350)
lstm_pred = np.array(lstm_names)
truth = splits['val']['label'].to_numpy()

lr_ok = lr_pred == truth
lstm_ok = lstm_pred == truth
agree = lr_pred == lstm_pred

print(f"1. Agree: {agree.mean():.1%}")
print(f"2. Acc when agree: {lr_ok[agree].mean():.1%}")
print(f"3. Disagree → LR acc: {lr_ok[~agree].mean():.1%} | LSTM acc: {lstm_ok[~agree].mean():.1%}")
print(f"4. Oracle: {(lr_ok | lstm_ok).mean():.1%} | LR: {lr_ok.mean():.1%} | LSTM: {lstm_ok.mean():.1%}")
print(f"5. LR errors in disagree rows: {(~agree[~lr_ok]).mean():.1%}")




1. Agree: 96.0%
2. Acc when agree: 93.8%
3. Disagree → LR acc: 36.1% | LSTM acc: 47.2%
4. Oracle: 93.4% | LR: 91.5% | LSTM: 91.9%
5. LR errors in disagree rows: 30.3%


In [76]:
val = splits["val"]
win = lstm_ok & ~lr_ok                     # LSTM right AND LR wrong
print("LSTM right, LR wrong:", win.sum())       # hint: count the Trues

sample = val[win].sample(5, random_state=42)
for idx, row in sample.iterrows():
    pos = val.index.get_loc(idx)              # row position → to look up predictions
    print("=" * 80)
    print(row["text_clean"][:300])
    print(f"TRUE: {row['label']}\nLR:   {lr_pred[pos]}\nLSTM: {lstm_pred[pos]}")


LSTM right, LR wrong: 2719
i am writing to formally express my concerns regarding zelle 's handling of disputes, partially in light of the recent cfpb lawsuit highlighting significant issues with this platform. the cfpb 's findings indicate that zelle has not only neglected to to protect consumers but has also allowed repeat 
TRUE: Money transfer, virtual currency, or money service
LR:   Credit reporting or other personal consumer reports
LSTM: Money transfer, virtual currency, or money service
i am writing to formally express my concern regarding zelles handling of disputes, particularly in light of the recent cfpb lawsuit highlighting significant issues with the platform. the cfpbs has neglected to protect consumers leaving users vulnerable to scams.
TRUE: Money transfer, virtual currency, or money service
LR:   Credit reporting or other personal consumer reports
LSTM: Money transfer, virtual currency, or money service
i have an account with redactedinfo redactedinfo redactedinfo red

In [77]:
import time, statistics, joblib

text = splits["val"]["Consumer complaint narrative"].iloc[0]       # raw text
vec_lr = joblib.load("../models/tfidf_b350.joblib")
lr     = joblib.load("../models/logreg_b350.joblib")

def lr_path(t):
    return lr.predict_proba(vec_lr.transform([first_350(clean_text(t))]))

def lstm_path(t):
    ids = vec_loaded(tf.constant([clean_text(t)]))
    return lstm(ids, training=False)            # direct call, not .predict()

for name, fn in [("LR", lr_path), ("LSTM", lstm_path)]:
    fn(text)                                     # warm-up 🔥
    times = []
    for _ in range(100):
        s = time.perf_counter(); fn(text); times.append(time.perf_counter() - s)
    print(f"{name}: median {statistics.median(times) * 1000:.1f} ms")


LR: median 1.3 ms
LSTM: median 1342.8 ms


In [ ]:
predict_one = tf.function(lambda ids: lstm(ids, training=False))

ids = vec_loaded(tf.constant([clean_text(text)]))     # same complaint as before
predict_one(ids)                                       # warm-up 🔥 (compiles, slow once)

def median_ms(fn, n=100):
    times = []
    for _ in range(n):
        s = time.perf_counter(); fn(); times.append(time.perf_counter() - s)
    return statistics.median(times) * 1000

vec_ms   = median_ms(lambda: vec_loaded(tf.constant([clean_text(text)])))
model_ms = median_ms(lambda: predict_one(ids))
print(f"Vectorise: {vec_ms:.1f} ms | Model: {model_ms:.1f} ms | Total: {vec_ms + model_ms:.1f} ms (eager was 1342.8)")


Vectorise: 4.4 ms | Model: 20.3 ms | Total: 24.7 ms (eager was 1342.8)


: 